# 🏢 Amazon ML 2026 — Business Entity Resolution
## GPU Accelerated Pipeline (50K Experiment)

---

### ✅ Prerequisites — Do These Before Running Any Cell

**Step 1 — Upload your ENTIRE project folder to Google Drive:**
It should look like this:
```
My Drive/
└── Amazon-ML-entity-resolution/
    ├── dataset/               ← with train/ and test/ inside
    ├── src/                   ← your code
    └── utils/
```

**Step 2 — Choose runtime:**
- Runtime → Change runtime type → **T4 GPU**

**Step 3 — Run cells top to bottom. Never skip a cell.**


## 🔧 SECTION 1: Setup

In [ ]:
# ── Cell 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted at /content/drive')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
# Colab already has numpy, pandas, scikit-learn, scipy.
# We only need lightgbm and rapidfuzz.
import subprocess, sys

pkgs = ['lightgbm>=4.3.0', 'rapidfuzz>=3.8.0', 'joblib>=1.4.0', 'cupy-cuda12x']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

print('✓ All dependencies installed')

# Verify
import lightgbm, rapidfuzz, joblib, sklearn, numpy, pandas, scipy
print(f'  lightgbm {lightgbm.__version__}  |  rapidfuzz {rapidfuzz.__version__}')
print(f'  sklearn {sklearn.__version__}  |  pandas {pandas.__version__}')

In [ ]:
# ── Cell 3: Configure all paths ────────────────────────────────────────────
import os
from pathlib import Path

# ┌─────────────────────────────────────────────────────────────────┐
# │  POINT THIS TO YOUR UPLOADED PROJECT FOLDER                     │
DRIVE_ROOT   = Path('/content/drive/MyDrive/Amazon-ML-entity-resolution')
DRIVE_DATA   = DRIVE_ROOT / 'dataset'
DRIVE_SRC    = DRIVE_ROOT / 'src'
DRIVE_OUTPUT = DRIVE_ROOT / 'output'
DRIVE_MODELS = DRIVE_ROOT / 'models'
DRIVE_CACHE  = DRIVE_ROOT / 'cache'
# └─────────────────────────────────────────────────────────────────┘

# Local working dir (fast ephemeral SSD — much faster than Drive for I/O)
LOCAL_ROOT   = Path('/content/workspace')
LOCAL_DATA   = LOCAL_ROOT / 'dataset'
LOCAL_OUTPUT = LOCAL_ROOT / 'output'
LOCAL_MODELS = LOCAL_ROOT / 'models'
LOCAL_SRC    = LOCAL_ROOT / 'src'

# Create local dirs
for d in [LOCAL_ROOT, LOCAL_OUTPUT, LOCAL_MODELS, LOCAL_SRC]:
    d.mkdir(parents=True, exist_ok=True)

# Create Drive dirs
for d in [DRIVE_OUTPUT, DRIVE_MODELS, DRIVE_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

print('Local workspace:', LOCAL_ROOT)
print('Drive root:     ', DRIVE_ROOT)


In [ ]:
# ── Cell 4: Auto-detect available RAM and set batch sizes ──────────────────
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print(f'Available RAM: {ram_gb:.1f} GB')

if ram_gb >= 40:
    print('  → High-memory instance detected (Colab Pro+)')
    S1_BATCH_SIZE  = 500
    S23_CHUNK_SIZE = 600_000
    K_CANDIDATES   = 100
elif ram_gb >= 20:
    print('  → Standard Pro instance (Colab Pro)')
    S1_BATCH_SIZE  = 300
    S23_CHUNK_SIZE = 400_000
    K_CANDIDATES   = 50
else:
    print('  → Free Colab instance — using conservative settings')
    S1_BATCH_SIZE  = 100
    S23_CHUNK_SIZE = 150_000   # peak ~240 MB per dense block
    K_CANDIDATES   = 50

print(f'\n  S1_BATCH_SIZE  = {S1_BATCH_SIZE}')
print(f'  S23_CHUNK_SIZE = {S23_CHUNK_SIZE:,}')
print(f'  K_CANDIDATES   = {K_CANDIDATES}')
print('\nPeak RAM per cosine block:',
      f'{S1_BATCH_SIZE * S23_CHUNK_SIZE * 4 / 1e9:.2f} GB')

In [ ]:
# ── Cell 5: Copy dataset and code to local SSD ─────────────────────────────
# We copy to /content/workspace because Drive I/O is too slow for ML workloads
import shutil

if DRIVE_DATA.exists():
    print('Copying dataset from Drive to local workspace (takes a minute)...')
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print('✓ Dataset copied')
else:
    print(f'⚠ Dataset not found at {DRIVE_DATA}')

if DRIVE_SRC.exists():
    shutil.copytree(DRIVE_SRC, LOCAL_SRC, dirs_exist_ok=True)
    print('✓ Source code copied from Drive')
else:
    print(f'⚠ Source code not found at {DRIVE_SRC}')

# ── Verify ────────────────────────────────────────────────────────────────
expected = [
    LOCAL_DATA / 'train' / 'train_source1.tsv',
    LOCAL_DATA / 'train' / 'train_source2.tsv',
    LOCAL_DATA / 'train' / 'train_source3.tsv',
    LOCAL_DATA / 'train' / 'train_ground_truth.tsv',
    LOCAL_DATA / 'test'  / 'test_source1.tsv',
    LOCAL_DATA / 'test'  / 'test_source2.tsv',
    LOCAL_DATA / 'test'  / 'test_source3.tsv',
]
print('\nData file check:')
all_ok = True
for f in expected:
    size = f'{f.stat().st_size/1e6:.0f} MB' if f.exists() else 'MISSING'
    icon = '✓' if f.exists() else '✗'
    print(f'  {icon} {f.name}: {size}')
    all_ok &= f.exists()
print('\n' + ('✅ All files present — ready to proceed!' if all_ok
               else '❌ Some files missing — fix before continuing'))


In [ ]:
# ── Cell 6: Add src to Python path and patch config for Colab ─────────────
import sys

src_path = str(LOCAL_SRC)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f'✓ Added to sys.path: {src_path}')

# Override config paths to point at local workspace
# (the src/config.py uses ROOT relative to the file — we patch after import)
import config as cfg
cfg.ROOT        = LOCAL_ROOT
cfg.TRAIN_DIR   = LOCAL_DATA  / 'train'
cfg.TEST_DIR    = LOCAL_DATA  / 'test'
cfg.OUTPUT_DIR  = LOCAL_OUTPUT
cfg.MODEL_DIR   = LOCAL_MODELS

# Apply RAM-aware batch sizes from Cell 4
cfg.S1_BATCH_SIZE  = S1_BATCH_SIZE
cfg.S23_CHUNK_SIZE = S23_CHUNK_SIZE
cfg.K_CANDIDATES   = K_CANDIDATES

print(f'✓ Config patched:')
print(f'  TRAIN_DIR      = {cfg.TRAIN_DIR}')
print(f'  MODEL_DIR      = {cfg.MODEL_DIR}')
print(f'  K_CANDIDATES   = {cfg.K_CANDIDATES}')
print(f'  S1_BATCH_SIZE  = {cfg.S1_BATCH_SIZE}')
print(f'  S23_CHUNK_SIZE = {cfg.S23_CHUNK_SIZE:,}')

## 📊 SECTION 2: EDA — Dataset Statistics

In [ ]:
# ── Cell 7: EDA ────────────────────────────────────────────────────────────
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

file_specs = [
    ('train_source1',    cfg.TRAIN_DIR / 'train_source1.tsv'),
    ('train_source2',    cfg.TRAIN_DIR / 'train_source2.tsv'),
    ('train_source3',    cfg.TRAIN_DIR / 'train_source3.tsv'),
    ('train_ground_truth', cfg.TRAIN_DIR / 'train_ground_truth.tsv'),
    ('test_source1',     cfg.TEST_DIR  / 'test_source1.tsv'),
    ('test_source2',     cfg.TEST_DIR  / 'test_source2.tsv'),
    ('test_source3',     cfg.TEST_DIR  / 'test_source3.tsv'),
]

for label, path in file_specs:
    df = pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)
    print(f'\n{label}: {len(df):,} rows')
    if 'country' in df.columns:
        print('  Countries:', df['country'].value_counts().to_dict())
    if 'matched_entity_ids' in df.columns:
        has = df['matched_entity_ids'].str.len() > 0
        counts = df['matched_entity_ids'].apply(lambda x: len(x.split(',')) if x else 0)
        print(f'  Has matches: {has.sum():,} | Singletons: {(~has).sum():,}')
        print(f'  Avg matches per entity: {counts.mean():.2f}')
    # NaN check (using actual NaN, not 'nan' string)
    for col in ['business_name', 'business_address']:
        if col in df.columns:
            nan_count = df[col].isin(['', 'nan', 'NaN', 'NAN']).sum()
            if nan_count > 0:
                print(f'  ⚠ {col}: {nan_count} empty/NaN values')
    del df

import gc; gc.collect()
print('\n✅ EDA complete')

## ⚙️ SECTION 3: Preprocessing

In [ ]:
# ── Cell 8: Preprocess and save all 6 source files ─────────────────────────
# Preprocessed files are saved as Parquet to Drive for fast reuse.
# If they already exist on Drive, they are loaded directly (skip reprocessing).
import time, gc
from preprocess import preprocess_df

CACHE_DIR = DRIVE_CACHE / 'preprocessed'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_or_preprocess(name: str, raw_path: Path, cache_dir: Path) -> pd.DataFrame:
    """Load preprocessed parquet if cached, otherwise process and save."""
    cache_path = cache_dir / f'{name}.parquet'
    if cache_path.exists():
        print(f'  Loading cached {name} from Drive...')
        return pd.read_parquet(cache_path)
    print(f'  Processing {name} ({raw_path.stat().st_size/1e6:.0f} MB)...')
    t = time.time()
    df = pd.read_csv(raw_path, sep='\t', dtype=str, keep_default_na=False)
    df = preprocess_df(df)
    df.to_parquet(cache_path, index=False)
    print(f'    ✓ Done in {time.time()-t:.0f}s → cached to {cache_path}')
    return df

print('=== Preprocessing (cached to Drive) ===')
t0 = time.time()

s1_train = load_or_preprocess('train_s1', cfg.TRAIN_DIR/'train_source1.tsv', CACHE_DIR)
s2_train = load_or_preprocess('train_s2', cfg.TRAIN_DIR/'train_source2.tsv', CACHE_DIR)
s3_train = load_or_preprocess('train_s3', cfg.TRAIN_DIR/'train_source3.tsv', CACHE_DIR)
s1_test  = load_or_preprocess('test_s1',  cfg.TEST_DIR /'test_source1.tsv',  CACHE_DIR)
s2_test  = load_or_preprocess('test_s2',  cfg.TEST_DIR /'test_source2.tsv',  CACHE_DIR)
s3_test  = load_or_preprocess('test_s3',  cfg.TEST_DIR /'test_source3.tsv',  CACHE_DIR)

print(f'\n✅ All preprocessed in {(time.time()-t0)/60:.1f} min')
print(f'   S1 train: {len(s1_train):,} | S2 train: {len(s2_train):,} | S3 train: {len(s3_train):,}')
print(f'   S1 test:  {len(s1_test):,}  | S2 test:  {len(s2_test):,}  | S3 test:  {len(s3_test):,}')

## 🔵 SECTION 4: Blocking — Candidate Generation

> ⚠️ **This is the slowest step** (~3-8 hours depending on Colab tier).  
> Progress is **checkpointed to Google Drive after each country bucket**.  
> If your session disconnects, just re-run from this cell — it will resume from the checkpoint.

In [ ]:
# ── Cell 9: Blocking with Drive checkpointing ──────────────────────────────
import time, gc, json, pickle
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path

BLOCKING_CACHE = DRIVE_CACHE / 'blocking'
BLOCKING_CACHE.mkdir(parents=True, exist_ok=True)

# ── Core blocking functions (country-bucketed, chunked cosine) ─────────────

def build_vectorizer(texts):
    vec = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 4),
        max_features=50_000, min_df=2,
        sublinear_tf=True, strip_accents=None, norm='l2', dtype=np.float32
    )
    mat = vec.fit_transform(texts)
    return vec, mat

def batched_topk_cosine(s1_mat, s23_mat, s23_ids, k,
                         s1_batch=S1_BATCH_SIZE, s23_chunk=S23_CHUNK_SIZE,
                         min_score=0.05, progress_prefix=''):
    n1, n23 = s1_mat.shape[0], s23_mat.shape[0]
    k = min(k, n23)
    top_ids    = [np.empty(0, dtype=object)    for _ in range(n1)]
    top_scores = [np.empty(0, dtype=np.float32) for _ in range(n1)]

    n_chunks = (n23 + s23_chunk - 1) // s23_chunk
    for ci, s23_start in enumerate(range(0, n23, s23_chunk)):
        s23_end   = min(s23_start + s23_chunk, n23)
        chunk_mat = s23_mat[s23_start:s23_end]
        chunk_ids = s23_ids[s23_start:s23_end]

        for s1_start in range(0, n1, s1_batch):
            s1_end = min(s1_start + s1_batch, n1)
            q_mat  = s1_mat[s1_start:s1_end]
            sims   = q_mat.dot(chunk_mat.T)
            if sp.issparse(sims): sims = sims.toarray()
            sims   = sims.astype(np.float32)

            for li in range(s1_end - s1_start):
                gi = s1_start + li
                row = sims[li]
                mask = row >= min_score
                if not mask.any(): continue
                merged_ids    = np.concatenate([top_ids[gi],    chunk_ids[mask]])
                merged_scores = np.concatenate([top_scores[gi], row[mask]])
                if len(merged_ids) > k:
                    idx = np.argpartition(merged_scores, -k)[-k:]
                else:
                    idx = np.arange(len(merged_ids))
                top_ids[gi]    = merged_ids[idx]
                top_scores[gi] = merged_scores[idx]

        pct = (ci+1)/n_chunks*100
        if (ci+1) % max(1, n_chunks//10) == 0 or ci == n_chunks-1:
            print(f'    {progress_prefix}chunk {ci+1}/{n_chunks} ({pct:.0f}%)')

    results = []
    for i in range(n1):
        if len(top_ids[i]) == 0:
            results.append([])
        else:
            order = np.argsort(top_scores[i])[::-1]
            results.append([(top_ids[i][j], float(top_scores[i][j])) for j in order])
    return results


def block_s1_vs_s23_with_checkpoint(
    s1_df, s23_df, source_tag, k, checkpoint_dir
):
    """
    Blocking with per-country checkpointing.
    Saves results per country to `checkpoint_dir/<source_tag>_<country>.pkl`.
    Resumes automatically if checkpoint exists.
    """
    all_cands = {eid: [] for eid in s1_df['entity_id']}
    countries = s1_df['country_norm'].unique()

    for country in sorted(countries):
        ckpt_path = checkpoint_dir / f'{source_tag}_{country.replace(" ","_")}.pkl'

        if ckpt_path.exists():
            print(f'  [{source_tag}] Country "{country}": loading checkpoint...')
            with open(ckpt_path, 'rb') as f:
                country_cands = pickle.load(f)
        else:
            s1_sub  = s1_df[s1_df['country_norm'] == country]
            s23_sub = s23_df[s23_df['country_norm'] == country]
            if len(s23_sub) == 0:
                s23_sub = s23_df  # global fallback
            print(f'  [{source_tag}] Country "{country}": '
                  f'{len(s1_sub):,} S1 vs {len(s23_sub):,} {source_tag}'
                  f' (K={k}, ETA ~{max(1, len(s1_sub)*len(s23_sub)//200_000_000)} min)')

            t = time.time()
            vec, s23_mat = build_vectorizer(s23_sub['blocking_text'].tolist())
            s1_vecs = vec.transform(s1_sub['blocking_text'].tolist())
            s23_ids = s23_sub['entity_id'].values

            hits = batched_topk_cosine(
                s1_vecs, s23_mat, s23_ids, k=k,
                progress_prefix=f'[{source_tag}/{country}] '
            )
            country_cands = {s1_id: hit for s1_id, hit
                             in zip(s1_sub['entity_id'], hits)}

            with open(ckpt_path, 'wb') as f:
                pickle.dump(country_cands, f)
            del vec, s23_mat, s1_vecs
            gc.collect()
            print(f'    ✓ Saved checkpoint → {ckpt_path.name}  '
                  f'({(time.time()-t)/60:.1f} min)')

        for s1_id, hits in country_cands.items():
            all_cands[s1_id].extend(hits)

    return all_cands


def run_full_blocking(s1_df, s2_df, s3_df, k, checkpoint_dir):
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    cands_s2 = block_s1_vs_s23_with_checkpoint(s1_df, s2_df, 'S2', k, checkpoint_dir)
    cands_s3 = block_s1_vs_s23_with_checkpoint(s1_df, s3_df, 'S3', k, checkpoint_dir)

    final = {}
    for s1_id in s1_df['entity_id']:
        score_map = {}
        for eid, score in cands_s2.get(s1_id, []):
            if score > score_map.get(eid, -1): score_map[eid] = score
        for eid, score in cands_s3.get(s1_id, []):
            if score > score_map.get(eid, -1): score_map[eid] = score
        final[s1_id] = [eid for eid, _ in sorted(score_map.items(), key=lambda x: -x[1])]
    return final

print('✓ Blocking functions defined')
print(f'  Checkpoint dir: {BLOCKING_CACHE}')

In [ ]:
# ── Cell 10: Run Train Blocking ────────────────────────────────────────────
# Generates candidates for training. Results saved to Drive per country.
# Re-running this cell is safe — completed countries are loaded from checkpoint.

TRAIN_BLOCK_DIR = BLOCKING_CACHE / 'train'
TRAIN_CANDIDATES_PATH = DRIVE_CACHE / 'train_candidates.pkl'

if TRAIN_CANDIDATES_PATH.exists():
    print('Loading existing train candidates from Drive...')
    with open(TRAIN_CANDIDATES_PATH, 'rb') as f:
        train_candidates = pickle.load(f)
    print(f'✓ Loaded {len(train_candidates):,} S1 entities')
else:
    t0 = time.time()
    print(f'=== TRAIN BLOCKING (K={K_CANDIDATES}) ===')
    print(f'S1={len(s1_train):,} | S2={len(s2_train):,} | S3={len(s3_train):,}')
    print('⏳ This will take 3-8 hours. Progress saved per country to Drive.\n')

    train_candidates = run_full_blocking(
        s1_train, s2_train, s3_train, K_CANDIDATES, TRAIN_BLOCK_DIR
    )

    # Save full merged result
    with open(TRAIN_CANDIDATES_PATH, 'wb') as f:
        pickle.dump(train_candidates, f)

    elapsed = (time.time() - t0) / 60
    n_with = sum(1 for v in train_candidates.values() if v)
    print(f'\n✅ Train blocking done in {elapsed:.1f} min')
    print(f'   Entities with candidates: {n_with:,}/{len(train_candidates):,}')

# Quick stats
avg_cands = np.mean([len(v) for v in train_candidates.values()])
print(f'   Avg candidates/S1: {avg_cands:.1f}')

## 🧠 SECTION 5: Feature Engineering + Training

In [ ]:
# ── Cell 11: Load GT, build train/val split ────────────────────────────────
import numpy as np
from features import build_feature_matrix, build_lookup, FEATURE_NAMES
from evaluate import score_f05

# Load ground truth
gt_raw = pd.read_csv(cfg.TRAIN_DIR/'train_ground_truth.tsv', sep='\t',
                     dtype=str, keep_default_na=False)
gt_dict = {
    row['source1_entity_id']: [x for x in row['matched_entity_ids'].split(',') if x]
    if row['matched_entity_ids'] else []
    for _, row in gt_raw.iterrows()
}
print(f'GT loaded: {len(gt_dict):,} entries')
del gt_raw; gc.collect()

# Stratified 80/20 split by country
RANDOM_STATE = 42
val_ids = set()
for country, grp in s1_train.groupby('country_norm'):
    n_val = max(1, int(len(grp) * 0.20))
    val_ids |= set(grp.sample(n=n_val, random_state=RANDOM_STATE)['entity_id'])

s1_train_split = s1_train[~s1_train['entity_id'].isin(val_ids)]
s1_val_split   = s1_train[s1_train['entity_id'].isin(val_ids)]
print(f'Train: {len(s1_train_split):,} | Val: {len(s1_val_split):,}')

gt_tr = {k: v for k, v in gt_dict.items() if k in set(s1_train_split['entity_id'])}
gt_vl = {k: v for k, v in gt_dict.items() if k in set(s1_val_split['entity_id'])}

In [ ]:
# ── Cell 12: Build labeled pairs + features (chunked, with Drive cache) ────
FEATURE_CACHE = DRIVE_CACHE / 'features'
FEATURE_CACHE.mkdir(parents=True, exist_ok=True)

X_train_path = FEATURE_CACHE / 'X_train.npy'
y_train_path = FEATURE_CACHE / 'y_train.npy'
X_val_path   = FEATURE_CACHE / 'X_val.npy'
y_val_path   = FEATURE_CACHE / 'y_val.npy'

NEG_PER_POS = 5

def build_pairs(s1_df, candidates, gt_dict, neg_per_pos=NEG_PER_POS, seed=42):
    rng = np.random.default_rng(seed)
    pairs, labels = [], []
    for s1_id in s1_df['entity_id']:
        cands  = candidates.get(s1_id, [])
        gt_set = set(gt_dict.get(s1_id, []))
        pos = [c for c in cands if c in gt_set]
        neg = [c for c in cands if c not in gt_set]
        # Include missed GT pairs (not in candidates) as positives still
        missed = [g for g in gt_set if g not in set(cands)]
        pos.extend(missed)
        for c in pos:
            pairs.append((s1_id, c)); labels.append(1)
        n_neg = min(len(neg), max(neg_per_pos * len(pos), 1))
        for c in (rng.choice(neg, n_neg, replace=False).tolist() if neg else []):
            pairs.append((s1_id, c)); labels.append(0)
    return pairs, labels

s23_lookup = build_lookup(pd.concat([s2_train, s3_train], ignore_index=True))

if X_train_path.exists() and y_train_path.exists():
    print('Loading cached feature matrices from Drive...')
    X_train = np.load(X_train_path)
    y_train = np.load(y_train_path)
    X_val   = np.load(X_val_path)
    y_val   = np.load(y_val_path)
else:
    print('Computing features...')
    t = time.time()

    s1_tr_lookup = build_lookup(s1_train_split)
    tr_pairs, tr_labels = build_pairs(s1_train_split, train_candidates, gt_tr)
    print(f'  Train pairs: {len(tr_pairs):,} ({sum(tr_labels):,} pos)')

    # Compute in chunks of 1M pairs to avoid RAM spikes
    CHUNK = 1_000_000
    X_chunks = []
    for i in range(0, len(tr_pairs), CHUNK):
        chunk = tr_pairs[i:i+CHUNK]
        X_chunks.append(build_feature_matrix(chunk, s1_tr_lookup, s23_lookup, n_jobs=-1))
        print(f'  Features: {min(i+CHUNK, len(tr_pairs)):,}/{len(tr_pairs):,}')
    X_train = np.vstack(X_chunks)
    y_train = np.array(tr_labels, dtype=np.int32)
    del X_chunks, s1_tr_lookup; gc.collect()

    s1_vl_lookup = build_lookup(s1_val_split)
    # Val candidates: use same train_candidates dict, filtered to val S1 entities
    vl_cands = {k: train_candidates.get(k, []) for k in s1_val_split['entity_id']}
    vl_pairs, vl_labels = build_pairs(s1_val_split, vl_cands, gt_vl)
    X_val = build_feature_matrix(vl_pairs, s1_vl_lookup, s23_lookup, n_jobs=-1)
    y_val = np.array(vl_labels, dtype=np.int32)

    np.save(X_train_path, X_train); np.save(y_train_path, y_train)
    np.save(X_val_path,   X_val);   np.save(y_val_path,   y_val)
    print(f'✅ Features done in {(time.time()-t)/60:.1f} min → cached to Drive')

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}')
print(f'Positives — train: {y_train.sum():,}  val: {y_val.sum():,}')

In [ ]:
# ── Cell 13: Compute blocking recall ──────────────────────────────────────
# Run this after blocking to understand recall ceiling
total_gt = sum(len(v) for v in gt_tr.values() if v)
found = sum(
    1 for s1_id, gt_ids in gt_tr.items()
    for g in gt_ids if g in set(train_candidates.get(s1_id, []))
)
print(f'Blocking recall (train): {found:,}/{total_gt:,} = {100*found/max(total_gt,1):.2f}%')
print(f'Miss rate: {100*(total_gt-found)/max(total_gt,1):.2f}% of true pairs never enter candidate set')
print('(These can NEVER be matched by the model — improve blocking to raise recall)')

In [ ]:
# ── Cell 14: Train LightGBM ────────────────────────────────────────────────
import lightgbm as lgb
import pickle

MODEL_PATH = DRIVE_MODELS / 'lgbm_model.pkl'
META_PATH  = DRIVE_MODELS / 'meta.pkl'

LGBM_PARAMS = dict(
    objective='binary', metric='auc',
    num_leaves=127, learning_rate=0.05,
    min_child_samples=20, subsample=0.80, colsample_bytree=0.80,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)

if MODEL_PATH.exists():
    print('Loading existing model from Drive...')
    with open(MODEL_PATH, 'rb') as f: model = pickle.load(f)
    with open(META_PATH,  'rb') as f: meta  = pickle.load(f)
    print(f'✓ Loaded  |  Val F₀.₅={meta["val_f05"]:.4f}  |  Threshold={meta["threshold"]}')
else:
    print('Training LightGBM...')
    dtrain = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_NAMES)
    dval   = lgb.Dataset(X_val,   label=y_val,   feature_name=FEATURE_NAMES,
                         reference=dtrain)
    model = lgb.train(
        LGBM_PARAMS, dtrain, num_boost_round=1000, valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=True),
            lgb.log_evaluation(100),
        ]
    )
    print('\n  Sweeping threshold on val F₀.₅...')
    val_probs = model.predict(X_val)

    # Rebuild val candidate list for proper per-entity scoring
    vl_cands_dict = {k: train_candidates.get(k, []) for k in s1_val_split['entity_id']}
    prob_map = {}
    vl_pairs2, _ = build_pairs(s1_val_split, vl_cands_dict, gt_vl)
    val_probs2 = model.predict(build_feature_matrix(vl_pairs2, build_lookup(s1_val_split), s23_lookup, n_jobs=-1))
    for (s1_id, s23_id), prob in zip(vl_pairs2, val_probs2):
        prob_map[(s1_id, s23_id)] = prob

    thr_grid = [round(t, 2) for t in [x*0.05 for x in range(4, 20)]]
    best_f05, best_thr = -1.0, 0.5
    for thr in thr_grid:
        pred = {s1_id: [c for c in vl_cands_dict.get(s1_id, []) if prob_map.get((s1_id,c),0) >= thr]
                for s1_id in s1_val_split['entity_id']}
        f = score_f05(pred, gt_vl)
        print(f'  thr={thr:.2f}  F₀.₅={f:.4f}')
        if f > best_f05: best_f05, best_thr = f, thr

    print(f'\n★ Best threshold={best_thr}  Val F₀.₅={best_f05:.4f}')
    meta = {'threshold': best_thr, 'val_f05': best_f05}
    with open(MODEL_PATH, 'wb') as f: pickle.dump(model, f)
    with open(META_PATH,  'wb') as f: pickle.dump(meta,  f)
    print(f'✓ Model saved → {MODEL_PATH}')

    # Feature importance
    imp = pd.DataFrame({'feature': FEATURE_NAMES,
                         'importance': model.feature_importance(importance_type='gain')
                       }).sort_values('importance', ascending=False)
    print('\nTop 10 feature importances:')
    print(imp.head(10).to_string(index=False))

## 🔮 SECTION 6: Test Set Prediction

In [ ]:
# ── Cell 15: Run blocking on TEST set ─────────────────────────────────────
TEST_BLOCK_DIR = BLOCKING_CACHE / 'test'
TEST_CANDIDATES_PATH = DRIVE_CACHE / 'test_candidates.pkl'

if TEST_CANDIDATES_PATH.exists():
    print('Loading test candidates from Drive...')
    with open(TEST_CANDIDATES_PATH, 'rb') as f:
        test_candidates = pickle.load(f)
else:
    print(f'=== TEST BLOCKING (K={K_CANDIDATES}) ===')
    print(f'S1={len(s1_test):,} | S2={len(s2_test):,} | S3={len(s3_test):,}')
    print('⏳ France will use its own TF-IDF bucket automatically.\n')
    t = time.time()
    test_candidates = run_full_blocking(
        s1_test, s2_test, s3_test, K_CANDIDATES, TEST_BLOCK_DIR
    )
    with open(TEST_CANDIDATES_PATH, 'wb') as f:
        pickle.dump(test_candidates, f)
    print(f'✅ Test blocking done in {(time.time()-t)/60:.1f} min')

all_test_pairs = [(s1_id, c) for s1_id, cands in test_candidates.items() for c in cands]
print(f'Total test candidate pairs: {len(all_test_pairs):,}')

In [ ]:
# ── Cell 16: Score test candidates + generate output TSVs ─────────────────
LOCAL_OUTPUT.mkdir(exist_ok=True)

threshold = meta['threshold']
print(f'Using threshold={threshold}  (val F₀.₅={meta["val_f05"]:.4f})')

# Build lookups
s1_test_lookup  = build_lookup(s1_test)
s23_test_lookup = build_lookup(pd.concat([s2_test, s3_test], ignore_index=True))

# Score in batches of 500K
BATCH = 500_000
prob_map_test = {}
for start in range(0, len(all_test_pairs), BATCH):
    batch = all_test_pairs[start:start+BATCH]
    X = build_feature_matrix(batch, s1_test_lookup, s23_test_lookup, n_jobs=-1)
    probs = model.predict(X)
    for (s1_id, s23_id), p in zip(batch, probs):
        prob_map_test[(s1_id, s23_id)] = p
    print(f'  Scored {min(start+BATCH, len(all_test_pairs)):,}/{len(all_test_pairs):,}')
    del X; gc.collect()

# Apply threshold
predictions = {}
for s1_id in s1_test['entity_id']:
    cands = test_candidates.get(s1_id, [])
    predictions[s1_id] = [c for c in cands if prob_map_test.get((s1_id,c),0) >= threshold]

# Save matching_results.tsv
rows_match = [{'source1_entity_id': s1_id,
               'matched_entity_ids': ','.join(predictions.get(s1_id, []))}
              for s1_id in s1_test['entity_id']]
pd.DataFrame(rows_match).to_csv(LOCAL_OUTPUT/'matching_results.tsv', sep='\t', index=False)

# Save candidate_pairs.tsv
rows_cand = [{'source1_entity_id': s1_id,
               'candidate_entity_ids': ','.join(test_candidates.get(s1_id, []))}
             for s1_id in s1_test['entity_id']]
pd.DataFrame(rows_cand).to_csv(LOCAL_OUTPUT/'candidate_pairs.tsv', sep='\t', index=False)

n_matched = sum(1 for v in predictions.values() if v)
print(f'\n✅ Output TSVs saved to {LOCAL_OUTPUT}')
print(f'   Matched: {n_matched:,} | Singletons: {len(predictions)-n_matched:,}')

## ✅ SECTION 7: Validate + Download

In [ ]:
# ── Cell 17: Copy outputs to Drive + validate ──────────────────────────────
import shutil, subprocess

# Copy to Drive for safe keeping
shutil.copy(LOCAL_OUTPUT/'matching_results.tsv',  DRIVE_OUTPUT/'matching_results.tsv')
shutil.copy(LOCAL_OUTPUT/'candidate_pairs.tsv',   DRIVE_OUTPUT/'candidate_pairs.tsv')
print(f'✓ TSVs saved to Drive: {DRIVE_OUTPUT}')

# Run validator (if utils/validate_submission.py is present)
# Copy validator from Drive if it's there
VALIDATOR_DRIVE = DRIVE_CODE / 'utils' / 'validate_submission.py'
VALIDATOR_LOCAL = LOCAL_ROOT / 'validate_submission.py'
if VALIDATOR_DRIVE.exists():
    shutil.copy(VALIDATOR_DRIVE, VALIDATOR_LOCAL)
    result = subprocess.run(
        ['python', str(VALIDATOR_LOCAL),
         '--matching',  str(LOCAL_OUTPUT/'matching_results.tsv'),
         '--candidate', str(LOCAL_OUTPUT/'candidate_pairs.tsv'),
         '--test-dir',  str(LOCAL_DATA/'test')],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode == 0:
        print('✅ VALIDATION PASSED — safe to submit!')
    else:
        print('❌ VALIDATION FAILED:')
        print(result.stderr)
else:
    print('⚠ Validator not found at', VALIDATOR_DRIVE)
    print('  Upload utils/validate_submission.py to Drive/AmazonML/code/utils/')

In [ ]:
# ── Cell 18: Download matching_results.tsv ─────────────────────────────────
# Downloads matching_results.tsv to your local machine.
# This is the file you upload to the leaderboard portal.
from google.colab import files

print('Downloading matching_results.tsv...')
files.download(str(LOCAL_OUTPUT / 'matching_results.tsv'))

# Optionally download candidate_pairs.tsv too (needed for final submission ZIP)
# files.download(str(LOCAL_OUTPUT / 'candidate_pairs.tsv'))

print('\n📊 Final stats:')
match_df = pd.read_csv(LOCAL_OUTPUT/'matching_results.tsv', sep='\t',
                        dtype=str, keep_default_na=False)
has_match = match_df['matched_entity_ids'].str.len() > 0
print(f'  Total rows:          {len(match_df):,}')
print(f'  With ≥1 match:       {has_match.sum():,}')
print(f'  Singletons (empty):  {(~has_match).sum():,}')
avg_m = match_df[has_match]['matched_entity_ids'].apply(lambda x: len(x.split(','))).mean()
print(f'  Avg matches per entity (non-singleton): {avg_m:.2f}')
print(f'\n  Val F₀.₅ (local estimate): {meta["val_f05"]:.4f}')
print(f'  Threshold used:            {meta["threshold"]}')

---
## 📋 Quick Reference: What Each Cell Does

| Cell | Action | Time | Checkpointed? |
|------|--------|------|---------------|
| 1 | Mount Google Drive | instant | — |
| 2 | Install deps (lightgbm, rapidfuzz) | ~1 min | — |
| 3 | Configure paths | instant | — |
| 4 | Auto-detect RAM, set batch sizes | instant | — |
| 5 | Extract ZIP / copy data | 5-10 min | — |
| 6 | Add src/ to Python path, patch config | instant | — |
| 7 | EDA — dataset stats | 5 min | — |
| 8 | **Preprocess all 6 files** | 10-20 min | ✅ Drive Parquet |
| 9 | Define blocking functions | instant | — |
| 10 | **TRAIN blocking (slowest)** | 3-8 hrs | ✅ Per-country PKL |
| 11 | Load GT, create train/val split | ~5 min | — |
| 12 | **Build feature matrices** | 1-3 hrs | ✅ Drive .npy |
| 13 | Print blocking recall | instant | — |
| 14 | **Train LightGBM + threshold sweep** | 20-30 min | ✅ Drive PKL |
| 15 | **TEST blocking** | 2-5 hrs | ✅ Per-country PKL |
| 16 | Score test candidates → output TSVs | 1-2 hrs | — |
| 17 | Validate format + copy to Drive | ~5 min | — |
| 18 | Download matching_results.tsv | instant | — |

---
## 🔄 If Your Session Disconnects

1. **Re-run cells 1-6** (setup — all fast)
2. **Re-run cell 8** — will load cached Parquet from Drive instantly
3. **Re-run cell 9** — defines functions
4. **Re-run cell 10** — will skip completed country checkpoints, resume from last one
5. Continue from where you left off

---
## 📁 Google Drive Structure After Full Run

```
My Drive/AmazonML/
├── student_resource.zip
├── code/
│   ├── src/                   ← your pipeline code
│   └── utils/validate_submission.py
├── cache/
│   ├── preprocessed/          ← .parquet files (6 files)
│   ├── blocking/
│   │   ├── train/             ← S2_us.pkl, S2_india.pkl, S3_us.pkl ...
│   │   └── test/              ← S2_us.pkl, S2_india.pkl, S2_france.pkl ...
│   ├── train_candidates.pkl   ← merged train blocking output
│   ├── test_candidates.pkl    ← merged test blocking output
│   └── features/
│       ├── X_train.npy, y_train.npy
│       └── X_val.npy,   y_val.npy
├── models/
│   ├── lgbm_model.pkl
│   └── meta.pkl               ← threshold + val F₀.₅
└── output/
    ├── matching_results.tsv   ← SUBMIT THIS
    └── candidate_pairs.tsv    ← include in final ZIP
```